In [1]:
!pip install ctgan --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.5/74.5 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 30.2 MB/s eta 0:00:00a 0:00:01


In [2]:
# ============================================================
# CELL 1: ALL IMPORTS, CONSTANTS, MODEL, UTILITIES
# Run this cell first after any session restart.
# ============================================================
import numpy as np
import torch
import torch.nn as nn
import math, joblib, warnings, gc
import pandas as pd
from ctgan import CTGAN
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.svm import OneClassSVM
from sklearn.metrics import roc_auc_score, average_precision_score, fbeta_score
from huggingface_hub import hf_hub_download

warnings.filterwarnings('ignore')
torch.backends.cudnn.benchmark = True

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)

# ── LOB layout ─────────────────────────────────────────────
K=100; M=144; D=128; D_MODEL=128; N_LAYERS=6; N_HEADS=8; D_FF=512; DROPOUT=0.1
N_LEVELS=10
ASK_PRICE=[4*l+0 for l in range(N_LEVELS)]
ASK_SIZE =[4*l+1 for l in range(N_LEVELS)]
BID_PRICE=[4*l+2 for l in range(N_LEVELS)]
BID_SIZE =[4*l+3 for l in range(N_LEVELS)]
BEST_ASK_P,BEST_ASK_S=0,1
BEST_BID_P,BEST_BID_S=2,3

# ── Model definition ───────────────────────────────────────
class PositionalEncoding(nn.Module):
    def __init__(self,d_model,max_len=500,dropout=0.1):
        super().__init__()
        self.dropout=nn.Dropout(dropout)
        pe=torch.zeros(max_len,d_model)
        pos=torch.arange(0,max_len).unsqueeze(1).float()
        div=torch.exp(torch.arange(0,d_model,2).float()*(-math.log(10000.0)/d_model))
        pe[:,0::2]=torch.sin(pos*div); pe[:,1::2]=torch.cos(pos*div)
        self.pe=pe.unsqueeze(0)
    def forward(self,x):
        return self.dropout(x+self.pe[:,:x.size(1)].to(x.device))

class BottleneckedTransformerAE(nn.Module):
    def __init__(self):
        super().__init__()
        self.input_proj=nn.Linear(M,D_MODEL)
        self.enc_pos=PositionalEncoding(D_MODEL,max_len=K+1,dropout=DROPOUT)
        enc_layer=nn.TransformerEncoderLayer(d_model=D_MODEL,nhead=N_HEADS,
                    dim_feedforward=D_FF,dropout=DROPOUT,batch_first=True)
        self.encoder=nn.TransformerEncoder(enc_layer,num_layers=N_LAYERS)
        self.bottleneck=nn.Linear(K*D_MODEL,D)
    def encode(self,x):
        e=self.enc_pos(self.input_proj(x))
        e=self.encoder(e)
        return self.bottleneck(e.flatten(start_dim=1))

# ── Load model weights ─────────────────────────────────────
tae_path=hf_hub_download(repo_id='Jaanusri/train',filename='best_transformer_ae.pt',repo_type='dataset')
model=BottleneckedTransformerAE().to(device)
checkpoint=torch.load(tae_path,map_location=device)
model.load_state_dict(checkpoint['model_state_dict'],strict=False)
model.float(); model.eval()
print('Transformer AE loaded')

# ── encode_windows (float32, chunked) ─────────────────────
def encode_windows(model, X, device, chunk_size=128):
    """
    float32 throughout — fp16 causes LayerNorm instability
    with pretrained weights. chunk_size=128 keeps peak VRAM
    well under 2GB. Reduce to 64 if you still see OOM.
    """
    z_list=[]
    model.float(); model.eval()
    with torch.no_grad():
        for start in range(0,len(X),chunk_size):
            end=min(start+chunk_size,len(X))
            chunk=torch.tensor(X[start:end],dtype=torch.float32).to(device)
            z_list.append(model.encode(chunk).cpu().numpy())
            del chunk; torch.cuda.empty_cache()
    return np.concatenate(z_list,axis=0)

# ── load_repo ──────────────────────────────────────────────
def load_repo(repo, window_size=100):
    batches=[]
    for i in range(20):
        try:
            fname=f'subseq_batch_{i}.npy'
            path=hf_hub_download(repo_id=repo,filename=fname,repo_type='dataset')
            batch=np.load(path)
            if batch.ndim==2: batch=batch.reshape(-1,window_size,144)
            if batch.ndim!=3: continue
            batches.append(batch)
            print(f'  Loaded {fname} -> {batch.shape}')
        except Exception: break
    if not batches: raise RuntimeError(f'No data from {repo}')
    return np.concatenate(batches,axis=0)

# ── FIX 3: max-pool score aggregation ─────────────────────
def point_scores_from_windows(win_scores, N, k):
    T=N+k-1; scores=np.full(T,-np.inf)
    for i,s in enumerate(win_scores):
        end=min(i+k,T); scores[i:end]=np.maximum(scores[i:end],s)
    scores[scores==-np.inf]=0.0
    return scores

def best_f4_threshold(scores, y_true, n=300):
    thresholds=np.percentile(scores,np.linspace(50,99.9,n))
    best_f4,best_tau=-1.0,thresholds[0]
    for tau in thresholds:
        f4=fbeta_score(y_true,(scores>tau).astype(int),beta=4,zero_division=0)
        if f4>best_f4: best_f4,best_tau=f4,tau
    return best_tau,best_f4

def recall_by_type(raw_scores, anom_idx, tau_win, n_anom):
    out={}
    for offset,name in enumerate(MTYPE_KEYS):
        idxs=[anom_idx[i] for i in range(offset,n_anom,3)]
        out[name]=float(np.mean(raw_scores[idxs]>tau_win)) if idxs else float('nan')
    return out

# ── Injection helpers ──────────────────────────────────────
def _local_stats(w,col,lookback=20):
    r=w[-lookback:,col]; return float(np.mean(r)),max(float(np.std(r)),1e-6)

def _corr_propagate(w,t_start,t_end,col_list,delta_t,decay=0.72):
    for l,col in enumerate(col_list):
        att=decay**l
        for i,t in enumerate(range(t_start,min(t_end,w.shape[0]))):
            w[t,col]+=att*delta_t[i]

def _enforce_no_cross(w):
    for l in range(N_LEVELS):
        ap,bp=ASK_PRICE[l],BID_PRICE[l]
        cx=w[:,bp]>w[:,ap]
        if cx.any():
            mid=(w[cx,ap]+w[cx,bp])/2.0
            w[cx,ap]=mid+1e-8; w[cx,bp]=mid-1e-8
        w[:,ASK_SIZE[l]]=np.maximum(w[:,ASK_SIZE[l]],0.0)
        w[:,BID_SIZE[l]]=np.maximum(w[:,BID_SIZE[l]],0.0)
    return w

def inject_pump_and_dump(window,rng):
    w=window.copy(); k=w.shape[0]
    pe=max(3,int(k*0.30)); de=max(pe+3,int(k*0.88))
    _,sap=_local_stats(w,BEST_ASK_P); _,sbp=_local_stats(w,BEST_BID_P)
    mas,_=_local_stats(w,BEST_ASK_S); mbs,_=_local_stats(w,BEST_BID_S)
    pd_=rng.uniform(1.2,2.0); vs=rng.uniform(0.4,0.9)
    tp=np.arange(pe)
    ra=pd_*sap*(tp/max(pe-1,1))**0.6; rb=pd_*sbp*(tp/max(pe-1,1))**0.6
    _corr_propagate(w,0,pe,ASK_PRICE,ra,0.75); _corr_propagate(w,0,pe,BID_PRICE,rb,0.75)
    for t in range(pe):
        f=(t+1)/pe; w[t,BEST_ASK_S]+=vs*abs(mas)*f; w[t,BEST_BID_S]+=vs*abs(mbs)*f
    nd=de-pe; td=np.arange(nd)
    da=pd_*sap*(1-(td/max(nd-1,1))**1.5); db=pd_*sbp*(1-(td/max(nd-1,1))**1.5)
    _corr_propagate(w,pe,de,ASK_PRICE,da,0.75); _corr_propagate(w,pe,de,BID_PRICE,db,0.75)
    dv=rng.uniform(0.5,1.0)
    for i,t in enumerate(range(pe,de)):
        f=(i+1)/nd; w[t,BEST_BID_S]-=dv*abs(mbs)*f; w[t,BEST_ASK_S]+=dv*abs(mas)*f
    return _enforce_no_cross(w)

def inject_layering(window,rng):
    w=window.copy(); k=w.shape[0]
    se=max(4,int(k*0.40)); no=rng.integers(8,13)
    _,sbp=_local_stats(w,BEST_BID_P); mbs,_=_local_stats(w,BEST_BID_S)
    pp=rng.uniform(0.8,1.5)*sbp; sv=rng.uniform(0.6,1.2)*abs(mbs)
    sp=max(2,se//no); cd=max(1,sp-1); bsd=np.zeros(k)
    for i in range(no):
        pt=min(i*sp,se-2); ct=min(pt+cd,se-1); f=(i+1)/no
        w[pt,BEST_BID_P]+=pp*f; bsd[pt]+=sv*(1.0+0.3*f); bsd[ct]-=sv*rng.uniform(0.7,1.0)
    _corr_propagate(w,0,se,BID_SIZE,bsd[:se],0.68)
    return _enforce_no_cross(w)

def inject_quote_stuffing(window,rng):
    w=window.copy(); k=w.shape[0]
    fs=rng.choice(['bid','ask'])
    sc=BEST_BID_S if fs=='bid' else BEST_ASK_S
    pc=BEST_BID_P if fs=='bid' else BEST_ASK_P
    dr=1 if fs=='bid' else -1
    ne=rng.integers(int(k*0.40),int(k*0.70)+1)
    st=rng.integers(0,max(1,k-ne)); et=min(st+ne,k)
    ms,ss=_local_stats(w,sc); _,sp=_local_stats(w,pc)
    rs=w[max(0,st-20):st,sc]
    if len(rs)<5: rs=w[:,sc]
    tiny=float(np.percentile(np.abs(rs),5))
    per=rng.integers(2,6); lg=abs(ms)+rng.uniform(0.3,0.8)*ss; po=rng.uniform(0.15,0.40)*sp
    for t in range(st,et):
        ph=(t-st)%per
        if ph==0:   w[t,sc]=lg;                       w[t,pc]+=dr*po
        elif ph==1: w[t,sc]=tiny*rng.uniform(0.01,0.08); w[t,pc]-=dr*po*rng.uniform(0.6,1.0)
    return _enforce_no_cross(w)

MANIPULATION_TYPES={'pump_and_dump':inject_pump_and_dump,
                    'layering':inject_layering,
                    'quote_stuffing':inject_quote_stuffing}
MTYPE_KEYS=list(MANIPULATION_TYPES.keys())

print('All definitions ready. Proceed to Cell 2.')


Device: cuda


best_transformer_ae.pt:   0%|          | 0.00/73.6M [00:00<?, ?B/s]

Transformer AE loaded
All definitions ready. Proceed to Cell 2.


In [ ]:
# ============================================================
# CELL 2: TRAIN OC-SVM (tuned nu) + CTGAN
# Only needs to run once per session.
# ============================================================
N_PCA=200; N_PER_TYPE=3000; CTGAN_EPOCHS=200
FEAT_COLS=[f'f_{i}' for i in range(N_PCA)]

# ── Load + encode training data ────────────────────────────
print('Loading training windows...')
X_train=load_repo('Jaanusri/test7',window_size=100)
print('Encoding...')
z_train=encode_windows(model,X_train,device,chunk_size=128)
torch.cuda.empty_cache(); gc.collect()

# ── Scaler ─────────────────────────────────────────────────
scaler=StandardScaler()
z_train_scaled=scaler.fit_transform(z_train)
print('Scaler fitted')

# ── FIX 4: OC-SVM nu sweep ────────────────────────────────
print('Training OC-SVM variants...')
NU_VALUES=[0.01,0.02,0.04,0.05,0.10]
ocsvm_models={}
for nu in NU_VALUES:
    clf=OneClassSVM(kernel='rbf',nu=nu,gamma='scale')
    clf.fit(z_train_scaled)
    ocsvm_models[nu]=clf
    print(f'  nu={nu} done')

del z_train,z_train_scaled; gc.collect()
print('OC-SVM training done. GPU memory freed.')

# ── Build CTGAN seed dataset ───────────────────────────────
print('\nBuilding CTGAN seed dataset...')
rng0=np.random.default_rng(seed=0)
wins,labs=[],[]
idx=rng0.choice(len(X_train),size=N_PER_TYPE,replace=False)
wins.append(X_train[idx]); labs+=['normal']*N_PER_TYPE
for mtype,fn in MANIPULATION_TYPES.items():
    idx=rng0.choice(len(X_train),size=N_PER_TYPE,replace=False)
    wins.append(np.array([fn(X_train[i],rng0) for i in idx]))
    labs+=[mtype]*N_PER_TYPE

all_w=np.concatenate(wins,axis=0)
flat=all_w.reshape(len(all_w),-1).astype(np.float32)
seed_labels=np.array(labs)
del all_w,wins,X_train; gc.collect()
print(f'Seed dataset: {flat.shape}')

# ── PCA ────────────────────────────────────────────────────
print('Fitting PCA...')
pca=PCA(n_components=N_PCA,random_state=42)
compressed=pca.fit_transform(flat)
print(f'Variance explained: {pca.explained_variance_ratio_.sum():.3f}')
del flat; gc.collect()

# ── Train CTGAN (CPU) ──────────────────────────────────────
print('\nTraining CTGAN (CPU, ~20-40 min)...')
df_ct=pd.DataFrame(compressed,columns=FEAT_COLS)
df_ct['label']=seed_labels
ctgan=CTGAN(epochs=CTGAN_EPOCHS,batch_size=500,
            generator_dim=(256,256,256),discriminator_dim=(256,256,256),
            generator_lr=1e-4,discriminator_lr=1e-4,
            cuda=False,verbose=True)
ctgan.fit(df_ct,discrete_columns=['label'])
# Quality check: compare real vs generated feature distributions
print('\nCTGAN quality check...')
for mtype in MTYPE_KEYS:
    real_df  = df_ct[df_ct['label']==mtype][FEAT_COLS].values
    fake_df  = ctgan.sample(300, condition_column='label',
                             condition_value=mtype)[FEAT_COLS].values
    # Mean absolute difference in feature means (should be low)
    mean_diff = np.abs(real_df.mean(axis=0) - fake_df.mean(axis=0)).mean()
    # Ratio of stds (should be close to 1.0)
    std_ratio = (fake_df.std(axis=0) / (real_df.std(axis=0) + 1e-8)).mean()
    print(f'  {mtype}: mean_diff={mean_diff:.4f}  std_ratio={std_ratio:.4f}')
print('  (good: mean_diff<0.5, std_ratio between 0.8 and 1.2)')
del df_ct,compressed,seed_labels; gc.collect()
print('CTGAN training complete.')

# ── Sampler ────────────────────────────────────────────────
def sample_ctgan_windows(ctgan,pca,mtype,n,window_shape=(100,144)):
    df=ctgan.sample(n,condition_column='label',condition_value=mtype)
    arr=pca.inverse_transform(df[FEAT_COLS].values)
    wins=arr.reshape(n,*window_shape)
    return np.array([_enforce_no_cross(w) for w in wins])

print('Sampler check:')
for m in MTYPE_KEYS:
    s=sample_ctgan_windows(ctgan,pca,m,n=2)
    print(f'  {m}: shape={s.shape} finite={np.isfinite(s).all()}')
print('\nAll training complete. Proceed to Cell 3.')


Loading training windows...


subseq_batch_0.npy:   0%|          | 0.00/576M [00:00<?, ?B/s]

  Loaded subseq_batch_0.npy -> (10000, 100, 144)


subseq_batch_1.npy:   0%|          | 0.00/576M [00:00<?, ?B/s]

  Loaded subseq_batch_1.npy -> (10000, 100, 144)


...

subseq_batch_4.npy:   0%|          | 0.00/576M [00:00<?, ?B/s]

  Loaded subseq_batch_4.npy -> (10000, 100, 144)


subseq_batch_5.npy:   0%|          | 0.00/310M [00:00<?, ?B/s]

  Loaded subseq_batch_5.npy -> (5379, 100, 144)
Encoding...
Scaler fitted
Training OC-SVM variants...
  nu=0.01 done
  nu=0.02 done
  nu=0.04 done
  nu=0.05 done
  nu=0.1 done
OC-SVM training done. GPU memory freed.

Building CTGAN seed dataset...
Seed dataset: (12000, 14400)
Fitting PCA...
Variance explained: 0.595

Training CTGAN (CPU, ~20-40 min)...


Gen. (-14.59) | Discrim. (-00.21):  98%|█████████▊| 195/200 [1:05:39<01:39, 19.97s/it]

In [ ]:
# ============================================================
# CELL 3: EVALUATION FUNCTION (all 5 fixes)
# ============================================================

def evaluate_repo_ctgan(X, repo_label, model, scaler, ocsvm_models,
                        ctgan, pca, device,
                        num_runs=3, anomaly_ratio=0.01, window_size=100):
    print('\n'+('='*57))
    print('EVALUATING: '+repo_label)
    print('='*57)
    N=len(X)
    print(f'Windows: {N:,}  anomaly_ratio={anomaly_ratio}')

    # Encode full repo once
    z_all=encode_windows(model,X,device,chunk_size=128)
    torch.cuda.empty_cache(); gc.collect()
    z_all_scaled=scaler.transform(z_all)

    # Pre-compute base scores for every nu
    base_scores_by_nu={nu:-clf.decision_function(z_all_scaled)
                       for nu,clf in ocsvm_models.items()}

    run_results=[]

    for run in range(num_runs):
        rng=np.random.default_rng(seed=42+run*13)
        n_anom=max(3,int(anomaly_ratio*N))
        anom_idx=rng.choice(N,size=n_anom,replace=False)

        # Sample CTGAN anomalies
        n_each=n_anom//3+2
        pool={m:sample_ctgan_windows(ctgan,pca,m,n=n_each) for m in MTYPE_KEYS}

        # Encode anomaly windows once
        anom_wins=np.stack([
            pool[MTYPE_KEYS[r%3]][r//3] if r//3<len(pool[MTYPE_KEYS[r%3]])
            else MANIPULATION_TYPES[MTYPE_KEYS[r%3]](X[idx],rng)
            for r,idx in enumerate(anom_idx)
        ])
        z_anom=encode_windows(model,anom_wins,device,chunk_size=128)
        torch.cuda.empty_cache(); gc.collect()
        z_anom_sc=scaler.transform(z_anom)

        # FIX 4: sweep nu, pick best F4
        best_f4_nu,best_nu=-1.0,list(ocsvm_models.keys())[0]
        for nu,clf in ocsvm_models.items():
            anom_sc=-clf.decision_function(z_anom_sc)
            raw=base_scores_by_nu[nu].copy()
            for i,idx in enumerate(anom_idx): raw[idx]=anom_sc[i]
            # FIX 5: normalise
            raw=(raw-raw.mean())/(raw.std()+1e-8)
            T_=N+window_size-1
            pt=point_scores_from_windows(raw,N,window_size)
            yt=np.zeros(T_,dtype=int)
            for idx in anom_idx: yt[idx:min(idx+window_size,T_)]=1
            _,f4_=best_f4_threshold(pt,yt)
            if f4_>best_f4_nu: best_f4_nu,best_nu=f4_,nu

        # Final eval with best nu
        anom_sc_best=-ocsvm_models[best_nu].decision_function(z_anom_sc)
        raw=base_scores_by_nu[best_nu].copy()
        for i,idx in enumerate(anom_idx): raw[idx]=anom_sc_best[i]
        # FIX 5: normalise
        raw=(raw-raw.mean())/(raw.std()+1e-8)
        T_=N+window_size-1
        pt=point_scores_from_windows(raw,N,window_size)
        yt=np.zeros(T_,dtype=int)
        for idx in anom_idx: yt[idx:min(idx+window_size,T_)]=1

        auc=roc_auc_score(yt,pt)
        auprc=average_precision_score(yt,pt)
        # FIX 2: optimal tau_win from F4 threshold
        tau_pt,f4=best_f4_threshold(pt,yt)
        pct=float(np.mean(pt<=tau_pt))*100
        tau_win=np.percentile(raw,pct)
        recalls=recall_by_type(raw,anom_idx,tau_win,n_anom)

        print(f'  Run {run+1}/{num_runs} [best nu={best_nu}]: AUROC={auc:.4f} AUPRC={auprc:.4f} F4={f4:.4f}')
        print(f'    PnD={recalls["pump_and_dump"]:.3f} Layer={recalls["layering"]:.3f} QS={recalls["quote_stuffing"]:.3f}')
        run_results.append(dict(auc=auc,auprc=auprc,f4=f4,
                               pnd=recalls['pump_and_dump'],
                               lay=recalls['layering'],qs=recalls['quote_stuffing']))
        del z_anom,z_anom_sc,raw,pt,yt; gc.collect(); torch.cuda.empty_cache()

    del z_all,z_all_scaled; gc.collect()

    print(f'  --- {repo_label} ---')
    print(f'  AUROC={np.mean([r["auc"] for r in run_results]):.4f} +/-{np.std([r["auc"] for r in run_results]):.4f}')
    print(f'  AUPRC={np.mean([r["auprc"] for r in run_results]):.4f} +/-{np.std([r["auprc"] for r in run_results]):.4f}')
    print(f'  F4   ={np.mean([r["f4"] for r in run_results]):.4f} +/-{np.std([r["f4"] for r in run_results]):.4f}')
    print(f'  R_PnD={np.nanmean([r["pnd"] for r in run_results]):.3f} R_Lay={np.nanmean([r["lay"] for r in run_results]):.3f} R_QS={np.nanmean([r["qs"] for r in run_results]):.3f}')

    return dict(AUROC=np.mean([r['auc'] for r in run_results]),
                AUROC_std=np.std([r['auc'] for r in run_results]),
                AUPRC=np.mean([r['auprc'] for r in run_results]),
                AUPRC_std=np.std([r['auprc'] for r in run_results]),
                F4=np.mean([r['f4'] for r in run_results]),
                F4_std=np.std([r['f4'] for r in run_results]),
                recall_pnd=np.nanmean([r['pnd'] for r in run_results]),
                recall_lay=np.nanmean([r['lay'] for r in run_results]),
                recall_qs =np.nanmean([r['qs']  for r in run_results]))

print('evaluate_repo_ctgan defined. Proceed to Cell 4.')


In [ ]:
# ============================================================
# CELL 4: MAIN RUN
# ============================================================
EVAL_REPOS=['Jaanusri/test7','Jaanusri/test8','Jaanusri/test9']
ANOMALY_RATIO=0.01   # FIX 1: 1% makes AUPRC meaningful
NUM_RUNS=3

all_results={}
all_windows=[]

for repo in EVAL_REPOS:
    X=load_repo(repo,window_size=100)
    all_windows.append(X.copy())
    res=evaluate_repo_ctgan(X,repo,model,scaler,ocsvm_models,
                            ctgan,pca,device,
                            num_runs=NUM_RUNS,anomaly_ratio=ANOMALY_RATIO)
    all_results[repo]=res
    del X; gc.collect(); torch.cuda.empty_cache()

if len(all_windows)>1:
    X_comb=np.concatenate(all_windows,axis=0)
    del all_windows; gc.collect()
    res=evaluate_repo_ctgan(X_comb,'COMBINED',model,scaler,ocsvm_models,
                            ctgan,pca,device,
                            num_runs=NUM_RUNS,anomaly_ratio=ANOMALY_RATIO)
    all_results['combined']=res
    del X_comb; gc.collect()

print('\n'+('='*60))
print('FINAL RESULTS  (CTGAN + 5 fixes)')
print('='*60)
print(f'{"Repo":<18} {"AUROC":>14} {"AUPRC":>14} {"F4":>10}')
print('-'*58)
for k,res in all_results.items():
    label=k.replace('Jaanusri/','') if k!='combined' else 'COMBINED'
    print(f'{label:<18} {res["AUROC"]:.4f}+/-{res["AUROC_std"]:.4f}  {res["AUPRC"]:.4f}+/-{res["AUPRC_std"]:.4f}  {res["F4"]:.4f}+/-{res["F4_std"]:.4f}')
print('-'*58)
print('Paper ref (LOBSTER): AUROC=0.960 AUPRC=0.842 F4=0.908')
print('Target (this setup): AUROC>0.78  AUPRC>0.70  F4>0.65')
